# 50 — Multi-head attention & positions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/050-multi-head-attention-positions.ipynb)

Attention mixes values according to query-key similarity. Here you compute the full weight matrix for three tokens, add position vectors and watch the matrix change, then split the same computation into two heads.

In [ ]:
import numpy as np                                          # array helpers
def attention(Q, K):                                        # weights for every query
    d = Q.shape[1]                                          # key dimension
    scores = Q @ K.T / np.sqrt(d)                           # scaled dot products
    scores = scores - scores.max(axis=1, keepdims=True)     # stable softmax shift
    e = np.exp(scores)                                      # exponentiate
    return e / e.sum(axis=1, keepdims=True)                # normalize each row
K = np.array([[1., 0.], [0., 1.], [1., 1.]])                # three keys
Q = K.copy()                                                # self-attention queries
V = np.array([[1., 0.], [0., 2.], [1., 1.]])                # three values
W = attention(Q, K)                                         # 3 x 3 weights
print(np.round(W, 3))                                       # show the matrix
print(np.round(W @ V, 3))                                   # mixed values
pos = np.array([[0., 0.], [1., 0.], [2., 0.]])              # simple positions
Wp = attention(Q + pos, K + pos)                            # same tokens, shifted
print(np.round(Wp, 3))                                      # compare the matrices

## Read the matrix

Each row of `W` sums to 1 and says how much that query pulls from each key. Token 1 matches key 1 exactly and key 2 not at all, so row 1 puts most of its weight on key 1. After adding `pos`, the same token comparisons produce different scores because the vectors have moved — that is position entering the mechanism.

`W @ V` is the output: each row is a weighted average of the three value vectors.

In [ ]:
import matplotlib.pyplot as plt                             # plotting library
fig, axes = plt.subplots(1, 2, figsize=(7, 3))              # two panels
axes[0].imshow(W, cmap="Blues", vmin=0, vmax=1)             # no position signal
axes[0].set_title("no position")                            # label
axes[1].imshow(Wp, cmap="Blues", vmin=0, vmax=1)            # with position added
axes[1].set_title("with position")                          # label
for ax in axes:                                             # both panels
    ax.set_xlabel("key")                                    # x label
    ax.set_ylabel("query")                                  # y label
plt.tight_layout()                                          # avoid overlap
plt.show()                                                  # render

## Multiple heads

A head is this whole computation in a narrower subspace. With `d_model = 512` and `h = 8`, each head works at width 64: project `Q`, `K` and `V` down, run attention, concatenate the eight outputs back to 512, and multiply by `W_O`. The parameter count is the same as one 512-wide head — the width is partitioned, not multiplied.

The heatmaps also show why one head is not enough: a single weight matrix can express one similarity pattern, while language needs several at once.

## Your turn

Add a causal mask by setting the upper triangle of `scores` to `-np.inf` before the softmax, and confirm each row still sums to 1. Then build two heads by projecting `Q`, `K`, `V` with two different random matrices, concatenate their outputs, and compare with the single-head result. Finally, remove `pos` and shuffle the token order — show that the outputs are only shuffled, not changed.